# Matched full-dataset LR continuation
Attach BTP-code-paired-full.zip, the complete epoch50 checkpoint ZIP and prepared public252 dataset. This notebook extracts the inputs automatically. Both arms train epochs51-60; only the LR schedule differs. Allow approximately2 hours plus setup/archiving. Download the complete final ZIP. Forced session termination can bypass the archive cell.

In [ ]:
from pathlib import Path
import shutil, zipfile, subprocess, sys
INPUTS = Path('/kaggle/input')
CODE = Path('/kaggle/working/BTP-paired')
def safe_extract(z, dest):
    dest = dest.resolve()
    for member in z.infolist():
        if not (dest / member.filename).resolve().is_relative_to(dest):
            raise ValueError('Unsafe ZIP member')
    z.extractall(dest)
if not CODE.exists():
    sources = list(INPUTS.rglob('paired_full_training.py'))
    if len(sources) == 1:
        shutil.copytree(sources[0].parent, CODE)
    else:
        found = []
        for archive in INPUTS.rglob('*.zip'):
            if zipfile.is_zipfile(archive):
                with zipfile.ZipFile(archive) as z:
                    if 'BTP/paired_full_training.py' in z.namelist(): found.append(archive)
        assert len(found) == 1, 'Attach exactly one BTP-code-paired-full.zip input.'
        staging = Path('/kaggle/working/paired_code_extract')
        staging.mkdir(exist_ok=False)
        with zipfile.ZipFile(found[0]) as z: safe_extract(z, staging)
        shutil.copytree(staging / 'BTP', CODE)
assert (CODE / 'paired_full_training.py').is_file()
expected = 'baseline_to_epoch50_20260917_204827'
refs = list(INPUTS.rglob(expected + '/model/last.pt'))
if not refs:
    for archive in INPUTS.rglob('*.zip'):
        if not zipfile.is_zipfile(archive): continue
        with zipfile.ZipFile(archive) as z:
            if expected + '/model/last.pt' in z.namelist():
                dest = Path('/kaggle/working/paired_reference')
                safe_extract(z, dest)
                refs = [dest / expected / 'model/last.pt']
                break
assert len(refs) == 1, 'Attach the complete epoch50 run ZIP or extracted folder.'
REFERENCE = refs[0].parent.parent
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
import torch
assert torch.cuda.is_available(), 'Enable the GPU.'
print('Code:', CODE, 'Reference:', REFERENCE, 'GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'verify_paired_full_training.py'], cwd=CODE, check=True)


In [ ]:
from datetime import datetime
from IPython.display import display, FileLink
OUT = Path('/kaggle/working') / ('paired_full_lr_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
try:
    subprocess.run([sys.executable, '-u', 'paired_full_training.py', '--root', str(DATA),
        '--reference', str(REFERENCE), '--out', str(OUT)], cwd=CODE, check=True)
finally:
    if OUT.exists():
        archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=OUT.name)
        with zipfile.ZipFile(archive) as z:
            assert z.testzip() is None
            missing = [f'{OUT.name}/{arm}/model/{name}' for arm in ['control','quarter_lr']
                for name in ['last.pt','best_iou.pth','best_psnr.pth']
                if f'{OUT.name}/{arm}/model/{name}' not in z.namelist()]
        print('Archive:', archive, 'Missing checkpoint files:', missing)
        display(FileLink(archive))
        print('Download the complete ZIP before closing the session. Check logs for successful epoch60 completion in BOTH arms.')


In [ ]:
import json
summary = json.loads((OUT / 'comparison_summary.json').read_text())
for arm, result in summary['arms'].items():
    print(arm, json.dumps({k:v for k,v in result.items() if k != 'epochs'}, indent=2))
assert set(summary['arms']) == {'control', 'quarter_lr'}, 'Comparison is incomplete.'
